# 03 · Supervised fine-tuning (SFT)

Starts from a pretrained rung and teaches it the chat format on smol-smoltalk (English, ~90%) and InstrucTurca (Turkish, ~10%). Only assistant tokens are trained. At every evaluation the pretraining validation sets are scored too, to measure forgetting.

| `RUN` | Starts from | Data | A100 time (estimate) |
|---|---|---|---:|
| `smoke` | `s50m-001` | 60 steps | ~5 min incl. compile |
| `s-50m` | `s50m-001` | 1 epoch | ~45 min |
| `m-100m` | `m100m-001` | 1 epoch | ~75 min |

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100***.

**Resumable** exactly like pretraining: rerun all cells with the same `RUN`. To stop training, use *Restart Kernel* and run all cells again; cell 2 removes the old kernel from the GPU.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
# In VS Code + Colab, "Restart Kernel" can leave the old kernel (and its compile workers)
# running and holding GPU memory, which later causes CUDA out-of-memory errors.
import os
import signal

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid == me:
        continue
    if "colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args):
        stale.append((pid, args[:80]))
for pid, args in stale:
    os.kill(pid, signal.SIGKILL)
    print("killed stale process", pid, args)
print("stale processes:", len(stale))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Run all tests

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 6. Download and verify the data

SFT windows (~1.2 GB) and, for the forgetting check, only the pretraining **validation** shards (~60 MB). Every file's SHA-256 is checked against its manifest.

In [ ]:
import json
import time

from huggingface_hub import snapshot_download

from glm_ladder.checkpoint import sha256_file

DATA_DIR = Path("/content/data")
started = time.perf_counter()
snapshot_download(repo_id=DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=token,
                  allow_patterns=["sft/*", "*/manifest.json", "*/val-*.bin"])
print(f"downloaded in {time.perf_counter() - started:.0f} s")

tokenizer_receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "tokenizer-16k.json").read_text())
sft_manifest = json.loads((DATA_DIR / "sft" / "manifest.json").read_text())
assert sft_manifest["tokenizer_sha256"] == tokenizer_receipt["sha256"], "SFT data: tokenizer mismatch"
for source, splits in sft_manifest["sources"].items():
    for split, info in splits.items():
        for file, digest in info["sha256"].items():
            assert sha256_file(DATA_DIR / "sft" / source / split / file) == digest, f"{source}/{split}/{file}"
        print(f"sft {source:8s} {split:5s} {info['windows']:>8,} windows  {info['conversations']:>8,} conversations")

PRETRAIN_SOURCES = ("english", "code", "turkish")
for source in PRETRAIN_SOURCES:
    manifest = json.loads((DATA_DIR / source / "manifest.json").read_text())
    for shard in manifest["shards"]:
        if shard["name"].startswith("val-"):
            assert sha256_file(DATA_DIR / source / shard["name"]) == shard["sha256"], shard["name"]
    print(f"pretraining {source:8s} validation ok")
print(f"verified in {time.perf_counter() - started:.0f} s")

## 7. Sources

The SFT sources are sampled in proportion to their size, so one pass over the data is about one epoch of each.

In [ ]:
from glm_ladder.loader import PackedSource
from glm_ladder.sft_data import SFTSource

SEQ_LEN = sft_manifest["seq_len"]
train_sources = {name: SFTSource(DATA_DIR / "sft" / name / "train") for name in sft_manifest["sources"]}
val_sources = {name: SFTSource(DATA_DIR / "sft" / name / "val") for name in sft_manifest["sources"]}
total_windows = sum(source.num_windows for source in train_sources.values())
weights = {name: source.num_windows / total_windows for name, source in train_sources.items()}
pretrain_val = {source: PackedSource(sorted(str(p) for p in (DATA_DIR / source).glob("val-*.bin")), SEQ_LEN)
                for source in PRETRAIN_SOURCES}
for name, source in train_sources.items():
    print(f"{name:8s} weight {weights[name]:.3f}  train windows {source.num_windows:,}  "
          f"val windows {val_sources[name].num_windows:,}")

## 8. Choose the run

Change only `RUN`. The pretrained weights are downloaded from the final checkpoint of the pretraining run and checked against the SHA-256 in its `trainer.json`.

SFT settings, the same for both rungs: batch 128 × 1024 tokens, peak learning rate 1e-4 (1/6 of pretraining) with 50 warmup steps and cosine decay, one epoch, fresh AdamW state.

In [ ]:
import math

import torch

from glm_ladder import get_preset
from glm_ladder.pretrain import PretrainConfig
from glm_ladder.runs import RunStore

RUN = "smoke"            # "smoke" -> "s-50m" -> "m-100m"

GPU = torch.cuda.get_device_name()
MICRO_BATCH = 32 if "A100" in GPU else 16
BATCH_ROWS = 128
EPOCH_TOKENS = total_windows * SEQ_LEN
common = dict(seq_len=SEQ_LEN, batch_rows=BATCH_ROWS, micro_batch=MICRO_BATCH, max_lr=1e-4, warmup_steps=50,
              eval_every=200, eval_windows=128, compile=True, checkpoint_minutes=15.0)
RUNS = {
    "smoke": ("s-50m", "s50m-001", PretrainConfig(run_name="sft-smoke-s50m-001", total_tokens=60 * BATCH_ROWS * SEQ_LEN,
                                                  **{**common, "warmup_steps": 10, "eval_every": 30})),
    "s-50m": ("s-50m", "s50m-001", PretrainConfig(run_name="sft-s50m-001", total_tokens=EPOCH_TOKENS, **common)),
    "m-100m": ("m-100m", "m100m-001", PretrainConfig(run_name="sft-m100m-001", total_tokens=EPOCH_TOKENS, **common)),
}
preset, base_run, cfg = RUNS[RUN]
model_config = get_preset(preset)

base_checkpoint = RunStore(api, RUNS_REPO, base_run).latest()
def download(name):
    return Path(api.hf_hub_download(repo_id=RUNS_REPO, filename=f"runs/{base_run}/{base_checkpoint}/{name}",
                                    repo_type="model"))
base_state = json.loads(download("trainer.json").read_text())
init_from = download("model.safetensors")
assert sha256_file(init_from) == base_state["model_sha256"], "pretrained weights do not match their SHA-256"
assert base_state["model_config"] == model_config.to_dict(), "preset differs from the pretrained model"

measured = {"s-50m": 166_500, "m-100m": 99_700}[preset]      # A100 tok/s from the pretraining receipts
print(f"{cfg.run_name}: from {base_run} @ {base_checkpoint}, {cfg.total_steps:,} steps x {cfg.tokens_per_step:,} tokens")
print(f"estimate on A100: {cfg.total_steps * cfg.tokens_per_step / measured / 60:.0f} min")

## 9. Train

Watch three numbers at each evaluation:

- **SFT validation loss** (`english`, `turkish`): should fall quickly in the first few hundred steps, then flatten.
- **Pretraining validation loss** (`extra`): may rise slightly; a large rise means the model is forgetting.
- The initial training loss: lower than pretraining, because assistant replies are easier to predict than web text.

In [ ]:
from glm_ladder.pretrain import run_pretraining

WORKDIR = Path("/content/runs")
store = RunStore(api, RUNS_REPO, cfg.run_name)
metadata = {"commit": commit, "preset": preset, "gpu": GPU, "data_repo": DATA_REPO, "stage": "sft",
            "init_from": f"{RUNS_REPO}/runs/{base_run}/{base_checkpoint}",
            "init_sha256": base_state["model_sha256"], "weights": weights}
result = run_pretraining(cfg, model_config, train_sources=train_sources, val_sources=val_sources,
                         weights=weights, store=store, workdir=WORKDIR, device=torch.device("cuda"),
                         metadata=metadata, init_from=init_from, extra_val_sources=pretrain_val)
print("finished:", result["finished"], "at step", result["step"])

## 10. Curves

In [ ]:
import matplotlib.pyplot as plt

from glm_ladder.runs import MetricsLog

rows = MetricsLog(WORKDIR / cfg.run_name / "metrics.jsonl").rows()
train = [row for row in rows if "loss" in row]
evals = [row for row in rows if "eval" in row]
steps = [row["step"] for row in evals]

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
left.plot([row["step"] for row in train], [row["loss"] for row in train], alpha=0.3, label="train (assistant tokens)")
for name in val_sources:
    left.plot(steps, [row["eval"][name] for row in evals], "o-", label=f"SFT val {name}")
left.set(xlabel="step", ylabel="cross-entropy", title=f"{cfg.run_name}: SFT loss")
left.legend()
for name in PRETRAIN_SOURCES:
    right.plot(steps, [row["eval_extra"][name] for row in evals], "o-", label=name)
right.set(xlabel="step", ylabel="cross-entropy", title="pretraining validation (forgetting)")
right.legend()
plt.tight_layout()
plt.show()

## 11. After the run

Close the runtime (the A100 keeps using units while idle). Locally:

```bash
python scripts/pretrain_receipt.py --run sft-s50m-001     # receipt
```